# **Testing and developing methods to integrate reward scoring**

In [1]:
import requests

import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras

import mlflow

/home/madbiker/miniconda3/envs/tf_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
I0000 00:00:1776979939.533473   19509 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
ORCHESTRATOR_URI = "http://localhost:8000"
DATA_PROCESSOR_1 = "http://localhost:8011"
DATA_PROCESSOR_2 = "http://localhost:8012"

TEST_FILE_PATH = "../02_data/course_test.npz"

In [3]:
test_data = np.load(TEST_FILE_PATH)
x_test = test_data["x_test"]
y_test = test_data["y_test"]

In [37]:
#Downloading global model:
global_model_resource = requests.get(f"{ORCHESTRATOR_URI}/get_model").json()
mlflow.set_tracking_uri(global_model_resource["model_host"])
global_model = mlflow.tensorflow.load_model(global_model_resource["model_uri"])

/home/madbiker/miniconda3/envs/tf_env/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [7]:
#Obtain a local update:
organizational_global_model_resource = requests.get(f"{ORCHESTRATOR_URI}/get_model_organizational").json()

model_request = {
    "global_model_host": organizational_global_model_resource["model_host"],
    "global_model_uri": organizational_global_model_resource["model_uri"],
    "local_model_id": "test",
    "train_data_uri": "/data/train_data.npz",
    "num_shares": 1
}

local_model_resource_1 = requests.post(f"{DATA_PROCESSOR_1}/train_model",
                                   json=model_request).json()
local_model_resource_2 = requests.post(f"{DATA_PROCESSOR_2}/train_model",
                                   json=model_request).json()

In [18]:
local_model_resource_1

{'model_server_host': 'http://dp1:5000',
 'model_share_uri': 'models:/m-103bbd6df0c84815b2e2787c9dcb9482',
 'n_samples': 391}

In [19]:
mlflow.set_tracking_uri("http://localhost:5001")
local_model_1 = mlflow.tensorflow.load_model(local_model_resource_1["model_share_uri"])

mlflow.set_tracking_uri("http://localhost:5002")
local_model_2 = mlflow.tensorflow.load_model(local_model_resource_2["model_share_uri"])

model_updates = {
    "model_1": local_model_1,
    "model_2": local_model_2
}
sample_counts = {
    "model_1": local_model_resource_1["n_samples"],
    "model_2": local_model_resource_2["n_samples"]
}

In [ ]:
def _measure_actual_contribution(local_model: keras.Sequential, global_model: keras.Sequential,
                                x_test: np.ndarray, y_test: np.ndarray) -> float:
    #TODO: compute global model loss only once!
    l_g = -global_model.evaluate(x_test, y_test)[0]
    l_l = -local_model.evaluate(x_test, y_test)[0]
    return l_l-l_g

def _round_normalization(contributions: dict[str|float], eps: float=1e-8) -> dict[str|float]:
    contrib_values = list(contributions.values())
    mean_contrib = np.mean(contrib_values)
    std_contrib = np.std(contrib_values)
    normalized_contributions = {k: (v - mean_contrib) / std_contrib
     for k, v in contributions.items()}
    return normalized_contributions

def _update_reputations(reputations: dict[str|float],
                       norm_contributions: dict[str|float], rho: float=0.8) -> dict[str|float]:
    for participant, contribution in norm_contributions.items():
        reputations[participant] = rho * reputations[participant] + (1 - rho) * contribution
    return reputations

def _adjust_reputations_with_confidence(reputations: dict[str|float],
                           participation_counts: dict[str|float],
                           lamda: float = 5) -> dict[str|float]:
    adjusted_reputations = {}
    for participant, reputation in reputations.items():
        count = participation_counts[participant]
        adjusted_reputations[participant] = reputation * count / (count + lamda)
    return adjusted_reputations

def _compute_delta(global_model: keras.Sequential, local_model: keras.Sequential) -> list[np.ndarray]:
    global_weights = global_model.get_weights()
    local_weights = local_model.get_weights()
    deltas = [local - glob for local, glob in zip(local_weights, global_weights)]
    return deltas

def _sigmoid(x):
        return 1 / (1 + np.exp(-x))

def _update_global_model(global_model: keras.Sequential, delta_server: list[np.ndarray]) -> keras.Sequential:
    global_weights = global_model.get_weights()
    updated_weights = [glob + delta for glob, delta in zip(global_weights, delta_server)]
    global_model.set_weights(updated_weights)
    return global_model

def delta_rep(model_updates: dict[str|keras.Sequential],
              global_model: keras.Sequential,
              x_test: np.ndarray, y_test: np.ndarray,
              n_samples: dict[str|float],
              a: float,
              participation_counts: dict[str|float],
              reputations: dict[str|float]) -> tuple[keras.Sequential, dict[str|float]]:

    contributions = {}
    for participant in model_updates:
        local_model = model_updates[participant]
        contribution = _measure_actual_contribution(local_model, global_model, x_test, y_test)
        contributions[participant] = contribution
    norm_contributions = _round_normalization(contributions)
    updated_reputations = _update_reputations(reputations, norm_contributions)
    adjusted_reputations = _adjust_reputations_with_confidence(
        updated_reputations,
        participation_counts
    )

    deltas = {participant: _compute_delta(global_model, model_updates[participant])
                for participant in model_updates}
    
    reputation_weights = {}
    for participant in model_updates:
        reputation_weights[participant] = n_samples[participant]*_sigmoid(a * adjusted_reputations[participant])

    total_weight = sum(reputation_weights.values())
    normalized_weights = {participant: weight / total_weight for participant, weight in reputation_weights.items()}
    print(normalized_weights[participant])
    print(deltas[participant])

    delta_server = None
    for participant in model_updates:
        weighted_delta = [normalized_weights[participant] * delta for delta in deltas[participant]]
        if delta_server is None:
            delta_server = weighted_delta
        else:
            delta_server = [d_s + w_d for d_s, w_d in zip(delta_server, weighted_delta)]
    global_model = _update_global_model(global_model, delta_server)
    return global_model, updated_reputations

In [34]:
participation_counts = {participant: 1 for participant in model_updates}
reputations = {participant: 0.5 for participant in model_updates}

new_model, new_reputations = delta_rep(model_updates,
          global_model,
          x_test, y_test,
          n_samples=sample_counts,
          a=5,
          participation_counts=participation_counts,
          reputations=reputations)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.5312 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.5632 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.5312 
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7541 - loss: 0.5086 
0.5748639095908386
[array([[ 0.01592863, -0.01827778, -0.00322416,  0.01054375,  0.00865285,
         0.02551244,  0.01924997, -0.01669423, -0.01553773, -0.04037128,
        -0.00614486,  0.01063848, -0.02249248,  0.02518124,  0.01948889,
        -0.01057571,  0.0263456 , -0.00971277, -0.00284922,  0.        ,
        -0.0256626 ,  0.        , -0.00254226, -0.03821974,  0.02749529,
         0.        ,  0.        ,  0.02843849, -0.01830813, -0.0226623 ,
        -0.00393644,  0.        ],
       [-0.01641577, -0.0010433 , -0.00296312,  0.01324039, -0.01762004,
        -0.0054837 , -0.02046311, -0.00226994,  0.00032198, -0.03249678,
         0.00505297, -0.00830714, -0.00127992, -0.02041099,  0.03

In [35]:
new_model.evaluate(x_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.5312 


[0.531190037727356, 0.7158470153808594]

In [38]:
global_model.evaluate(x_test, y_test)

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7158 - loss: 0.6135  


[0.6135109066963196, 0.7158470153808594]